# Reproducing Figure xx: GRABACh3.0 Acetylcholine Biosensor Analysis

This notebook reanalyses raw image files and reproduces Figure x from **Belal et al. 2025** TITLE TBC.

**Dataset**: DANDI:xxxxx - Title TBC

**Analysis approach**:
- **Biosensor**: GRABACh3.0 genetically encoded acetylcholine indicator
- **Stimulation**: Single-pulse electrical stimulation to evoke acetylcholine release


In [ ]:
# Imports + config
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pynwb import NWBHDF5IO

sys.path.append(str(Path.cwd() / "Python functions"))
from master_functions import *

import warnings
warnings.filterwarnings(
    "ignore",
    message="Schema conflict.*ndx-optogenetics.*",
)

SAVE = False
EXPORT_DIR = Path.cwd() / "Paper analysis" / "Figure 7" / "xlsx"
NWB_DIR = Path.cwd() / "NWBdata" / "001832"

pmt_background = 144
baseline_window = 1
xlim = [4.5, 7]

nwb_index = load_dataset_description_index(NWB_DIR)
nwb_files = sorted(path for path in nwb_index if path.exists() and path.name.endswith("_ophys.nwb"))
print(f"Found {len(nwb_files)} NWB files")


In [ ]:
# Load first NWB frames, draw ROI

%matplotlib widget
from matplotlib.widgets import RectangleSelector

nwb_files = sorted(path for path in nwb_index if path.exists() and path.name.endswith("_ophys.nwb"))
print(f"Found {len(nwb_files)} NWB files")

# load frames from first file only — for ROI drawing
with NWBHDF5IO(str(nwb_files[0]), "r", load_namespaces=True) as io:
    nwb = io.read()
    raw_keys = [k for k in nwb.acquisition if "TwoPhotonSeries" in k]
    sensor_keys = [k for k in raw_keys if "GRABCh" in k or "GaAsP" in k]
    frames = nwb.acquisition[sensor_keys[0]].data[:]

roi_coords = {}

fig, ax = plt.subplots()
ax.imshow(frames.mean(axis=0), cmap="gray")
ax.set_title("Draw ROI — rectangle will be converted to ellipse")

def on_select(eclick, erelease):
    r0, c0 = int(eclick.ydata), int(eclick.xdata)
    r1, c1 = int(erelease.ydata), int(erelease.xdata)
    roi_coords.update(r0=min(r0,r1), r1=max(r0,r1),
                      c0=min(c0,c1), c1=max(c0,c1))
    print("ROI selected:", roi_coords)

sel = RectangleSelector(ax, on_select, useblit=True, button=[1], interactive=True)
plt.show()



In [ ]:
# Build ellipse mask, extract fluorescence from all NWB files

if not roi_coords:
    raise RuntimeError("No ROI drawn — draw a rectangle in Cell 2 first")

cy = (roi_coords["r0"] + roi_coords["r1"]) / 2
cx = (roi_coords["c0"] + roi_coords["c1"]) / 2
ry = (roi_coords["r1"] - roi_coords["r0"]) / 2
rx = (roi_coords["c1"] - roi_coords["c0"]) / 2
Y, X = np.ogrid[:frames.shape[1], :frames.shape[2]]
mask = ((Y - cy)/ry)**2 + ((X - cx)/rx)**2 <= 1
print(f"Ellipse mask: {mask.sum()} pixels")

records = []
for nwb_path in nwb_files:
    meta = {**nwb_index.get(nwb_path, {}), **parse_filename_meta(nwb_path)}
    if not meta:
        print(f"  skip (unrecognised name): {nwb_path.name}")
        continue

    with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
        nwb = io.read()
        raw_keys = [k for k in nwb.acquisition if "TwoPhotonSeries" in k]
        sensor_keys = [k for k in raw_keys if "GRABCh" in k or "GaAsP" in k]
        if not sensor_keys:
            print(f"  skip (no sensor series): {nwb_path.name}")
            continue

        series = nwb.acquisition[sensor_keys[0]]
        frm = series.data[:]

        # 1) acquisition timestamps (image frames)
        try:
            ts_img = np.asarray(series.get_timestamps(), dtype=float)
        except Exception:
            rate = float(series.rate)
            start = float(series.starting_time or 0.0)
            ts_img = start + np.arange(frm.shape[0]) / rate

        # 2) align to RoiResponseSeries timeline
        ts = ts_img.copy()
        if "ophys" in nwb.processing and "Fluorescence" in nwb.processing["ophys"].data_interfaces:
            fluor = nwb.processing["ophys"]["Fluorescence"]
            roi_keys = [k for k in fluor.roi_response_series.keys()
                        if ("GRABCh" in k or "GaAsP" in k)]
            if roi_keys:
                roi_ts = np.asarray(fluor.roi_response_series[roi_keys[0]].get_timestamps(), dtype=float)
                ts = ts_img + (roi_ts[0] - ts_img[0])

        # stim time from trials table
        stim_time = np.nan
        if nwb.trials is not None:
            trials_df = nwb.trials.to_dataframe()
            if "stimulus_start_time" in trials_df.columns:
                st = pd.to_numeric(trials_df["stimulus_start_time"], errors="coerce").dropna()
                if len(st):
                    stim_time = float(st.iloc[0])

        f_roi = frm[:, mask].mean(axis=1)

    records.append({
        **meta,
        "nwb_file":    nwb_path.name,
        "nwb_path":    str(nwb_path),
        "fluorescence": f_roi,
        "timestamps":   ts,
        "stim_time":    stim_time,
        "n_frames":     len(f_roi),
    })
    print(f"  {nwb_path.name}: {len(f_roi)} frames, stim={stim_time:.2f}s")

df = pd.DataFrame(records)
if not df.empty:
    df = df.sort_values("nwb_file").reset_index(drop=True)
    df["run_id"] = (
        df.groupby(["group", "animal_id", "slice_id", "roi_id"])
          .cumcount()
        + 1
    )
print(f"\n{len(df)} records loaded")


In [ ]:
# Compute dF/F0
rows = []
for _, trial in df.iterrows():
    t         = np.asarray(trial["timestamps"], dtype=float)
    f_raw     = np.asarray(trial["fluorescence"], dtype=float)
    stim_time = float(trial["stim_time"]) - t[0]
    dff       = compute_dff(f_raw=f_raw, time=t, stim_time=stim_time,
                            pmt_background=pmt_background, baseline_window=baseline_window)
    row = trial.to_dict()
    row["fluorescence"] = dff
    rows.append(row)

df_fl = pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
ii = 1
plot_dff(time=df_fl.loc[ii, "timestamps"], dff=df_fl.loc[ii, "fluorescence"], stim_time=stim_time, xlim=xlim, lwd=1.0, tick_len=4, axis_color="black")

In [ ]:
#  Average repeats
group_cols = ["group", "animal_id", "slice_id", "roi_id"]
rows = []

for keys, grp in df_fl.groupby(group_cols, sort=False):
    traces = [np.asarray(f, dtype=float) for f in grp["fluorescence"].values]
    min_len = min(len(tr) for tr in traces)
    dff_avg = np.vstack([tr[:min_len] for tr in traces]).mean(axis=0)
    ref = grp.sort_values("run_id").iloc[0].to_dict()
    ref["fluorescence"] = dff_avg
    ref["timestamps"]   = np.asarray(ref["timestamps"], dtype=float)[:min_len]
    ref["n_reps"]       = len(grp)
    rows.append(ref)

df_fl_average = pd.DataFrame(rows).reset_index(drop=True)


In [ ]:
# Peak dF / F0 + boxplot
     
df_fl_average["fmax"] = df_fl_average["fluorescence"].apply(
    lambda f: np.asarray(f, dtype=float).max()
)

ctrl = df_fl_average[df_fl_average["group"] == "ctrl"]
test = df_fl_average[df_fl_average["group"] == "test"]

fig, ax = plt.subplots()

box_plot = boxplot_rtype(
    ax,
    [ctrl["fmax"].values, test["fmax"].values],
    rtype=7,
    whis="minmax",
    positions=[1, 2],
    patch_artist=True,
    widths=0.25,
    boxprops=dict(linewidth=1.5),
    whiskerprops=dict(linewidth=1.5),
    capprops=dict(linewidth=1.5),
    median_overhang = 0.02,
    medianprops=dict(color="black", linewidth=4),
    showpoints=True,
    jitter_frac=1.2,
    point_diameter=4,
    point_alpha=1,
    point_color="gray",
    point_edgecolor="black",
    paired=False,
)

box_plot["boxes"][0].set_facecolor("white")
box_plot["boxes"][1].set_facecolor("white")

ax.set_xticks([1, 2])
ax.set_xticklabels(["ctrl", "MCI-Park"])
ax.set_ylabel("peak ΔF/F (fmax)")
ax.set_title("Peak fluorescence per ROI")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
ctrl["fmax"].values

In [ ]:
test["fmax"].values

In [ ]:
df